In [1]:
import numpy as np #handling array objects
import os #doing the file seeking to find the right location in the file
import matplotlib.pyplot as plt #plotting
from scipy.optimize import curve_fit
import tqdm
import scipy

In [81]:
def readInOutputFile(outputFilename):
    optionsStructType = np.dtype([
        ('B0', np.float64, 3),
        ('E', np.float64, 3),
        ('L', np.float64, 3),
        ('yi', np.float64, 3),
        ('posHistBins', np.float64, 3),
        ('m', np.float64),
        ('t0', np.float64),
        ('tf', np.float64),
        ('rtol', np.float64),
        ('atol', np.float64),
        ('beta', np.float64),
        ('uround', np.float64),
        ('safe', np.float64),
        ('fac1', np.float64),
        ('fac2', np.float64),
        ('hmax', np.float64),
        ('hmin', np.float64),
        ('h', np.float64),
        ('T', np.float64),
        ('gamma', np.float64),
        ('V', np.float64),
        ('a', np.float64),
        ('w', np.float64),
        ('swapStepSize', np.float64),
        ('ioutInt', np.float64),
        ('nmax', np.uint32),
        ('integratorType', np.int32),
        ('numParticles', np.int32),
        ('numPerGPUBlock', np.int32),
        ('iout', np.int32),
        ('numPhiBins', np.int32),
        ('numThetaBins', np.int32),
        ('dist', 'S', 1),
        ('output', 'S', 1),
        ('gas_coll', bool),
        ('diffuse', bool),
        ('gravity', bool),
        ('fixedStepSize', bool),
        ('keepStepSize', bool),
        ('pad', 'S', 5) #this is padding space just designed to fix the structure padding done in C++
    ])
    file = open(outputFilename, 'rb')
    parameters = np.fromfile(file, count=1, dtype=optionsStructType)[0]
    data = None
    if parameters['output'] == b'n':
        #in this case it was the full dump of all particle data
        outputDtype = np.dtype([
            ('t', '<f8'),
            ('xx', '<f8'),
            ('xy', '<f8'),
            ('xz', '<f8'),
            ('vx', '<f8'),
            ('vy', '<f8'),
            ('vz', '<f8'),
            ('sx', '<f8'),
            ('sy', '<f8'),
            ('sz', '<f8')])
        file.seek(0, 0)
        data = np.fromfile(file, dtype=outputDtype, offset=parameters.nbytes)
        numTimes = data.shape[0]//parameters['numParticles']
        numPer = int((parameters['tf']-parameters['t0'])/parameters['ioutInt'])+1
        data = data[:numTimes*parameters['numParticles']].reshape(-1, parameters['numParticles']).T #this returns the data in a little more convenient format, I think
    elif parameters['output'] == b'h':
        #this is the histogram output format instead now
        numx = int(parameters['L'][0]/parameters['gridSize'])+1
        numy = int(parameters['L'][1]/parameters['gridSize'])+1
        numz = int(parameters['L'][2]/parameters['gridSize'])+1

        numVecBins = int(np.ceil(2.0/parameters['vecBinSize']))

        xbins = np.arange(-parameters['L'][0]/2.0, parameters['L'][0]/2.0, parameters['gridSize'])
        ybins = np.arange(-parameters['L'][0]/2.0, parameters['L'][0]/2.0, parameters['gridSize']) 
        zbins = np.arange(-parameters['L'][0]/2.0, parameters['L'][0]/2.0, parameters['gridSize'])
        sbins = np.arange(-1.0, 1.0, parameters['vecBinSize'])
        outputDtype = np.dtype([
            ('t', np.float64),
            ('x', np.uint32, numx),
            ('y', np.uint32, numy),
            ('z', np.uint32, numz),
            ('sx', np.uint32, numVecBins),
            ('sy', np.uint32, numVecBins),
            ('sz', np.uint32, numVecBins)
        ])
        file.seek(0, 0)
        data = np.fromfile(file, dtype=outputDtype, offset=parameters.nbytes)
        data = {'xbins': xbins,
               'ybins': ybins,
               'zbins': zbins,
               'sbins': sbins,
               'data': data}
    elif parameters['output'] == b'a':
        outputDtype = np.dtype([
            ('t', np.float64),
            ('sx', np.float64),
            ('dsx', np.float64),
            ('sy', np.float64),
            ('dsy', np.float64),
            ('sz', np.float64),
            ('dsz', np.float64)
        ])
        file.seek(0, 0)
        data = np.fromfile(file, dtype=outputDtype, offset=parameters.nbytes)
    file.close()
    return parameters, data

def shiftDatapoints(phis):
    shifted = np.copy(phis)
    initHist, bins = np.histogram(phis, bins = 1000) #calculate the initial histogram and then adjust based on the mode
    peakloc = np.argmax(initHist)
    location = bins[peakloc]
    #shift everything to be close to that peak
    lowLocs = shifted<=(location-np.pi)
    highLocs = shifted>=(location+np.pi)
    shifted[lowLocs] += 2.0*np.pi
    shifted[highLocs] -= 2.0*np.pi
    mean = np.nanmean(shifted)
    std = np.nanstd(shifted)
    shifted[shifted<(mean-std*5)] = np.nan
    shifted[shifted>(mean+std*5)] = np.nan
    return shifted

def analyzeDataset(dset, randomSample = False, plot=False, coord1 = 'sy', coord2 = 'sz'):
    times = np.sort(np.unique(dset[0]['t']))
    phis = np.arctan2(dset[coord1], dset[coord2])
    for i in range(len(times)):
        if randomSample:
            phis[:,i] = shiftDatapoints(np.random.choice(phis[:,i], size=phis[:,i].shape))
        else:
            phis[:,i] = shiftDatapoints(phis[:,i])
        if plot:
            plt.hist(phis[:,i], bins = 1000)
            plt.title(times[i])
            plt.yscale('log')
            plt.show()
    means = np.nanmean(phis, axis=0)
    stds = np.nanstd(phis, axis=0)
    means = np.abs(means)
    return times, means, stds

def writeParameterFile(filename, dist = 'C', V = '5.0', T = '0.4', gas_coll = 'false', diffuse = 'true',
                       rtol = '1.0e-14', atol = '1.0e-14', gravity = 'true', particle='neutron',
                       B0 = '3.0e-6, 0, 0', E = '75.0e5, 0, 0', h = '0.0001',
                       hmin = '1.0e-9', hmax='1.0',
                       numParticles = '16384', numPerGPUBlock = '256', t0 = '0.0', tf = '0.1',
                       ioutInt = '0.0001', integratorType = '2', output = 'n', a = '0.0', w = '0.0',
                      keepStepSize = 'false'):
    with open(filename, 'w') as f:
        f.write('dist, '+dist + '\n')
        f.write('V, '+V + '\n')
        f.write('T, '+T + '\n')
        f.write('gas_coll, '+gas_coll + '\n')
        f.write('diffuse, '+diffuse + '\n')
        f.write('rtol, '+rtol + '\n')
        f.write('atol, '+atol + '\n')
        f.write('hmax, '+hmax+'\n')
        f.write('hmin, '+hmin+'\n')
        f.write('gravity, '+gravity + '\n')
        if particle == 'neutron':
            gamma = '-1.832471850e8'
            m = '1.674927e-27'
            f.write('gamma, '+gamma + '\n')
            f.write('m, '+m + '\n')
        elif particle == 'helium-3':
            gamma = '-2.037947093e8'
            m = '1.20200437865e-26'
            f.write('gamma, '+gamma + '\n')
            f.write('m, '+m)
        f.write('B0, '+B0 + '\n')
        f.write('E, '+E + '\n')
        f.write('h, '+h + '\n')
        f.write('numParticles, '+numParticles + '\n')
        f.write('numPerGPUBlock, '+numPerGPUBlock + '\n')
        f.write('t0, '+t0 + '\n')
        f.write('tf, '+tf + '\n')
        f.write('ioutInt, '+ioutInt + '\n')
        f.write('integratorType, '+integratorType + '\n')
        f.write('output, '+output + '\n')
        f.write('a, '+a+'\n')
        f.write('w, '+w+'\n')  
        f.write('keepStepSize, '+keepStepSize+'\n')
    return

In [117]:
filename = 'neutronTest.txt'
outDir = '/storage/home/hcoda1/9/dmathews33/scratch/spinDressing/initialTest/'
particle = 'neutron'
V='1.0'
T='0.4'
gas_coll='false'
diffuse='false'
gravity='true'
B0 = '3.0e-6, 0, 0'
E = '75.0e5, 0, 0'
numParticles = '128'
t0='0.0'
tf='100.0'
ioutInt='50.0'
a='0.0'
w='0.0'
h='0.001'
keepStepSize = 'false'

integratorType='0'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, 
                   a=a, w=w, keepStepSize=keepStepSize)
outname = outDir + 'initTestDOP.out'
!~/Spin-tracking/cpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsDOPCPU = readInOutputFile(outname)

integratorType='0'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, 
                   a=a, w=w, keepStepSize=keepStepSize)
outname = outDir + 'initTestDOP.out'
!~/Spin-tracking/gpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsDOPGPU = readInOutputFile(outname)

initializing
initialized
0, 50, 4199
1, 100, 4004
8234
initializing
initialized
0, 50, 24088
1, 100, 23990
50125


In [118]:
integratorType='1'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, 
                   a=a, w=w, keepStepSize=keepStepSize)
outname = outDir + 'initTestRK45.out'
!~/Spin-tracking/cpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsRK45CPU = readInOutputFile(outname)



integratorType='1'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, 
                   a=a, w=w, keepStepSize=keepStepSize)
outname = outDir + 'initTestRK45.out'
!~/Spin-tracking/gpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsRK45GPU = readInOutputFile(outname)

initializing
initialized
0, 50, 136
1, 100, 100
267
initializing
initialized
0, 50, 931
1, 100, 913
2162


In [119]:
integratorType='2'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, a=a, w=w)
outname = outDir + 'initTestMagnus.out'
!~/Spin-tracking/cpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsMagnusCPU = readInOutputFile(outname)

integratorType='2'
writeParameterFile(filename, particle='neutron', V=V, T=T, gas_coll=gas_coll, diffuse=diffuse, 
                   gravity=gravity, B0=B0, E=E, numParticles=numParticles, 
                   t0=t0, tf=tf, ioutInt=ioutInt, integratorType=integratorType, a=a, w=w)
outname = outDir + 'initTestMagnus.out'
!~/Spin-tracking/gpuTestmain ~/Spin-tracking/scripts/neutronTest.txt {outname}
resultsMagnusGPU = readInOutputFile(outname)

initializing
initialized
0, 50, 483
1, 100, 380
877
initializing
initialized
0, 50, 1358
1, 100, 1357
2969


In [122]:
for name in resultsDOPCPU[1].dtype.names:
    a = resultsDOPCPU[1][name]
    b = resultsMaGPU[1][name]
    print(name, np.allclose(a, b))

t True
xx True
xy True
xz True
vx True
vy True
vz True
sx True
sy True
sz True


In [73]:
resultsMagnusCPU[1]['t']

array([[0.  , 0.01, 0.02, ...,  nan,  nan,  nan],
       [0.  , 0.01, 0.02, ...,  nan,  nan,  nan],
       [0.  ,  nan,  nan, ...,  nan,  nan,  nan],
       ...,
       [0.  , 0.01, 0.02, ...,  nan,  nan,  nan],
       [0.  , 0.01, 0.02, ...,  nan,  nan,  nan],
       [0.  , 0.01, 0.02, ...,  nan,  nan,  nan]])

DOP: CPU and GPU Agree

RK45: CPU and GPU all agree with the original DOP CPU results

Magnus: neither CPU nor GPU agree with DOP
    CPU and GPU don't agree either
    Appears to be the NaN issue causing this
        When I mask NaN values this disappears, but they don't share the same NaN locations which is strange
    When masking NaN values it also agrees with the other methods, but we shouldn't be seeing NaN to begin with